# Ders 8: İleri Pekiştirmeli Öğrenme ve RLHF

**İleri Derin Öğrenme** — Haydar Kılıç

Ön koşul: *Derin Öğrenme*, Ders 9 (MDP'ler, Q-öğrenme, REINFORCE).

Giriş dersi, yakınsamanın garanti olduğu tablo tabanlı pekiştirmeli öğrenmeyi kapsıyordu. Bu defter,
fonksiyon yaklaşımı devreye girdiğinde nelerin bozulduğunu — **ölümcül üçlü**, aşırı tahmin yanlılığı
ve politika gradyanı varyansı — ele alıyor; sonra modern dil modeli eğitimine giden hattı takip
ediyor: TRPO'nun güven bölgesi, PPO'nun kırpılmış amaç fonksiyonu, insan tercihlerinden uydurulan
ödül modelleri ve DPO'nun kapalı form kısayolu.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt

np.random.seed(0)
plt.rcParams["figure.dpi"] = 100
softmax = lambda z, ax=-1: np.exp(z - z.max(ax, keepdims=True))/np.exp(z - z.max(ax, keepdims=True)).sum(ax, keepdims=True)
print("Kütüphaneler yüklendi.")


## 1. Ölümcül Üçlü

Tablo tabanlı TD öğrenme yakınsar. Şu üç bileşen bir araya geldiğinde ise yakınsamak zorunda değildir:

1. **Fonksiyon yaklaşımı** (durumlar parametre paylaşır),
2. **Önyükleme (bootstrapping)** (hedef, modelin kendi kestirimini içerir),
3. **Politika-dışı (off-policy)** eğitim (veri dağılımı, değerlendirilen politikadan farklıdır).

Nedeni şudur: TD hiçbir amaç fonksiyonu üzerinde gradyan inişi değildir — $r + \gamma \hat v(s')$
hedefi, aynı parametrelere bağlı olmasına rağmen sabit gibi ele alınır. Politika-dışı örnekleme
altında elde edilen güncellemenin bir büzülme (contraction) olduğu garanti değildir ve ağırlıklar
geometrik olarak ıraksayabilir. Aşağıda Tsitsiklis–Van Roy karşı örneğinin en yalın hâli var:
öznitelik paylaşan iki durum, politika-dışı bir dağılım ve her yerde sıfır ödül — yani gerçek değer
fonksiyonu $v \equiv 0$ ve görülen her ıraksama tamamen algoritmik bir başarısızlıktır.


In [ ]:
# Tsitsiklis-Van Roy iki durumlu karşı örneği.
# Tek skaler ağırlık w; öznitelikler phi(s0)=1, phi(s1)=2; her geçiş s1'de biter; tüm ödüller 0.
# Gerçek değer fonksiyonu v = 0'dır; |w|'deki her büyüme algoritmik ıraksamadır.
phi   = np.array([1.0, 2.0])
gamma = 0.99

def run(mu, rule="td", steps=400, lr=0.02):
    w, hist = 1.0, []
    rng = np.random.default_rng(0)
    for _ in range(steps):
        s = 0 if rng.random() < mu else 1            # politika-dışı örnekleme dağılımı
        delta = 0.0 + gamma*phi[1]*w - phi[s]*w      # TD hatası, sonraki durum her zaman s1
        if rule == "td":
            w += lr*delta*phi[s]                     # yarı-gradyan TD(0)
        else:
            w += lr*delta*(phi[s] - gamma*phi[1])    # Bellman artığının gerçek gradyanı
        hist.append(abs(w)*np.linalg.norm(phi))
    return np.array(hist)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for mu, c in zip([0.02, 0.1, 0.5, 0.9], ["#c6dbef", "#6baed6", "#2171b5", "#08306b"]):
    axes[0].semilogy(run(mu), lw=2, c=c, label=f"P(s0 örnekle) = {mu}")
axes[0].set_xlabel("güncelleme"); axes[0].set_ylabel("||v_tahmin||   (gerçek değer 0)")
axes[0].set_title("Politika-dışı örnekleme altında yarı-gradyan TD")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3, which="both")

axes[1].semilogy(run(0.5, "td"),       lw=2, label="yarı-gradyan TD  (ıraksıyor)")
axes[1].semilogy(run(0.5, "residual"), lw=2, label="Bellman artığı gradyanı  (yakınsıyor)")
axes[1].set_xlabel("güncelleme"); axes[1].set_ylabel("||v_hat||")
axes[1].set_title("Çözüm, onu dürüst bir gradyan yöntemine çevirmek")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3, which="both")
plt.tight_layout(); plt.show()

for mu in [0.02, 0.1, 0.5, 0.9]:
    h = run(mu)
    print(f"P(s0)={mu:4.2f}: ||v_tahmin|| {h[0]:.2f} -> {h[-1]:.3e}")
print("\nPratikteki çözümler: hedef ağlar (önyüklemeyi dondur), veriyi politikaya yakın tutan tekrar")
print("bellekleri ve yakınsama kanıtı olan gradyan-TD yöntemleri.")


## 2. Aşırı Tahmin Yanlılığı ve Çift Q-Öğrenme

Q-öğrenmenin hedefi $r + \gamma \max_{a'} Q(s',a')$'dir. Her $Q(s',a')$ üzerindeki gürültü
**yansız** olsa bile maksimum yukarı yönlü yanlıdır:

$$\mathbb{E}\left[\max_a \hat Q(a)\right] \;\ge\; \max_a \mathbb{E}\left[\hat Q(a)\right],$$

çünkü Jensen eşitsizliğine göre maksimum dışbükeydir. Yanlılık eylem sayısıyla ve gürültü düzeyiyle
birlikte büyür ve önyükleme onu tüm değer fonksiyonuna yayar.

**Çift Q-öğrenme**, bu maksimumun iki rolünü ayırır: bir kestirici argmax'ı **seçer**, ikincisi onu
**değerlendirir**. Hataları bağımsızsa, seçim hatası artık değerlendirmeyi şişirmez.


In [ ]:
def bias_experiment(n_actions, noise=1.0, trials=20000, seed=0):
    rng = np.random.default_rng(seed)
    true_q = np.zeros(n_actions)                                   # tüm eylemler eşit derecede iyi
    qa = true_q + noise*rng.normal(size=(trials, n_actions))
    qb = true_q + noise*rng.normal(size=(trials, n_actions))
    single = qa.max(1)                                             # gürültülü bir kestirimin maksimumu
    double = qb[np.arange(trials), qa.argmax(1)]                   # A ile seç, B ile değerlendir
    return single.mean(), double.mean()

n_list = [2, 4, 8, 16, 32, 64, 128]
single = [bias_experiment(n)[0] for n in n_list]
double = [bias_experiment(n)[1] for n in n_list]

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].semilogx(n_list, single, "o-", lw=2, base=2, label="maks Q  (tek kestirici)")
axes[0].semilogx(n_list, double, "s-", lw=2, base=2, label="çift Q-öğrenme")
axes[0].axhline(0, ls="--", c="k", lw=1, label="gerçek değer = 0")
axes[0].set_xlabel("eylem sayısı"); axes[0].set_ylabel("durumun kestirilen değeri")
axes[0].set_title("Aşırı tahmin eylem sayısıyla büyüyor")
axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

noises = np.linspace(0.05, 2.0, 20)
axes[1].plot(noises, [bias_experiment(16, s)[0] for s in noises], "o-", lw=2, label="tek")
axes[1].plot(noises, [bias_experiment(16, s)[1] for s in noises], "s-", lw=2, label="çift")
axes[1].axhline(0, ls="--", c="k", lw=1)
axes[1].set_xlabel("kestirim gürültüsü std"); axes[1].set_ylabel("kestirilen değer")
axes[1].set_title("...ve gürültü düzeyiyle (16 eylem)")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

# Önyükleme ile yayılım: yanlılık zincir boyunca birikir
gam = 0.95
chain = np.zeros(60); chain_d = np.zeros(60)
b_single, b_double = bias_experiment(16)[0], bias_experiment(16)[1]
for i in range(1, 60):
    chain[i]   = gam*chain[i-1]   + b_single
    chain_d[i] = gam*chain_d[i-1] + b_double
axes[2].plot(chain, lw=2, label="tek kestirici")
axes[2].plot(chain_d, lw=2, label="çift kestirici")
axes[2].axhline(0, ls="--", c="k", lw=1)
axes[2].set_xlabel("önyükleme derinliği"); axes[2].set_ylabel("birikmiş değer hatası")
axes[2].set_title("Önyükleme adım başına yanlılığı biriktirir")
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"16 eylem, birim gürültü: tek kestirici {b_single:+.3f}   çift {b_double:+.3f}   (gerçek 0)")


## 3. Politika Gradyanı Varyansı, Temel Değerler ve GAE

REINFORCE kestiricisi $\nabla J = \mathbb{E}[\nabla \log \pi(a\mid s)\, G]$ yansızdır ve çoğu zaman
kullanışsızdır: varyansı, eylemin ne kadar iyi olduğuyla değil getirinin *büyüklüğüyle* ölçeklenir.
Duruma bağlı herhangi bir **temel değer** (baseline) $b(s)$ çıkarmak yansızlığı bozmaz; çünkü
$\mathbb{E}[\nabla \log \pi(a|s)] = 0$'dır. Varyansı minimize eden seçim $V(s)$'ye yakındır ve bu da
avantajı verir: $A = G - V$.

**GAE** ardından tüm yanlılık–varyans ailesini tek bir parametreyle tarar:

$$\hat A_t^{\text{GAE}(\lambda)} = \sum_{l\ge0} (\gamma\lambda)^l \delta_{t+l},
\qquad \delta_t = r_t + \gamma V(s_{t+1}) - V(s_t).$$

$\lambda = 0$ tek adımlı TD avantajıdır (düşük varyans, $V$ yanlışsa yanlı); $\lambda = 1$ Monte
Carlo'dur (yansız, yüksek varyans).


In [ ]:
# 1B oyuncak: ödül eyleme zayıf bağlı, artı büyük bir sabit kayma
def grad_samples(offset, baseline, n=4000, seed=0):
    rng = np.random.default_rng(seed)
    a = rng.normal(0, 1, n)                       # politika N(0,1), theta = ortalama
    r = offset + 2.0*a                            # avantaj a'da doğrusal
    return (a)*(r - baseline)                     # birim std'li Gauss için grad log pi(a) = a

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
offsets = [0, 5, 20, 100]
for off in offsets:
    g_nob = grad_samples(off, 0.0)
    g_bas = grad_samples(off, off)
    axes[0].scatter([off], [g_nob.std()], c="crimson", s=45)
    axes[0].scatter([off], [g_bas.std()], c="seagreen", s=45)
axes[0].scatter([], [], c="crimson", label="temel değer yok")
axes[0].scatter([], [], c="seagreen", label="temel değer = V(s)")
axes[0].set_xlabel("sabit ödül kayması"); axes[0].set_ylabel("gradyan kestiriminin std'si")
axes[0].set_title("Sabit bir kayma saf varyans ekler"); axes[0].legend(fontsize=9); axes[0].grid(alpha=0.3)

axes[1].hist(grad_samples(20, 0.0), bins=60, alpha=0.6, color="crimson", label="temel değer yok")
axes[1].hist(grad_samples(20, 20.0), bins=60, alpha=0.6, color="seagreen", label="temel değer ile")
axes[1].axvline(2.0, c="k", ls="--", lw=1.5, label="gerçek gradyan")
axes[1].set_title("İkisi de yansız; ama yalnızca biri kullanılabilir"); axes[1].legend(fontsize=9)

# Kasten kusurlu bir değer fonksiyonuyla GAE yanlılık-varyans taraması
def gae_experiment(lam, gamma=0.99, T=50, n=3000, v_err=1.0, seed=0):
    rng = np.random.default_rng(seed)
    r = rng.normal(1.0, 1.0, (n, T))
    V_true = np.array([sum(gamma**k for k in range(T-t)) for t in range(T+1)])
    V_hat  = V_true + v_err*rng.normal(size=T+1)                  # yanlı değer fonksiyonu
    delta  = r + gamma*V_hat[1:][None, :] - V_hat[:-1][None, :]
    w = (gamma*lam)**np.arange(T)
    A = np.array([ (delta[:, t:] * w[:T-t]).sum(1) for t in range(T)]).T
    G = np.array([ (r[:, t:] * gamma**np.arange(T-t)).sum(1) for t in range(T)]).T
    A_true = G - V_true[:-1][None, :]
    return np.abs(A.mean(0) - A_true.mean(0)).mean(), A.std(0).mean()

lams = np.linspace(0, 1, 21)
bias, var = zip(*[gae_experiment(l) for l in lams])
axes[2].plot(lams, bias, "o-", lw=2, label="|yanlılık|")
axes[2].plot(lams, var, "s-", lw=2, label="standart sapma")
axes[2].set_xlabel("GAE lambda"); axes[2].set_title("lambda yanlılığı varyansla takas eder")
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print("Tipik pratik: gamma = 0.99, lambda = 0.95 -- varyansın çoğu düşer, yanlılık az kalır.")


## 4. Güven Bölgeleri: TRPO'dan PPO'ya

Bir politika gradyanı adımı yalnızca yerel olarak geçerlidir: vekil amaç fonksiyonu
$\mathbb{E}[\frac{\pi_\theta(a|s)}{\pi_{\text{eski}}(a|s)} A]$, gerçek iyileşmeyi ancak politikalar
birbirine yakın kaldığı sürece yaklaşıklar. Çok büyük bir adım atıldığında $\pi_{\text{eski}}$ ile
toplanan veri yeni politikayı artık tarif etmez; pekiştirmeli öğrenmenin tek bir güncellemede
çökebilmesinin nedeni budur.

**TRPO**, kısıtlı ikinci mertebe bir adımla
$\mathbb{E}[\text{KL}(\pi_{\text{eski}} \| \pi_\theta)] \le \delta$ kısıtını uygular. **PPO** ise
neredeyse aynı etkiyi birinci mertebe bir hileyle elde eder: olasılık oranı $r_t(\theta)$ kırpılır,
böylece oran $[1-\epsilon, 1+\epsilon]$ aralığından çıktığında amaç fonksiyonu düzleşir:

$$L^{\text{CLIP}} = \mathbb{E}\Big[\min\big(r_t A_t,\ \text{clip}(r_t, 1-\epsilon, 1+\epsilon) A_t\big)\Big].$$

İncelikli olan kısım $\min$'dir: sınırı **kötümser** yapar; böylece yanlış yöne giden bir oran
kırpılmaz ve düzeltici bir gradyan üretmeye devam eder.


In [ ]:
r = np.linspace(0, 2.5, 500)
eps = 0.2

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
for A, ax, name in [(1.0, axes[0], "pozitif avantaj A = +1"), (-1.0, axes[1], "negatif avantaj A = -1")]:
    unclipped = r*A
    clipped   = np.clip(r, 1-eps, 1+eps)*A
    obj = np.minimum(unclipped, clipped)
    ax.plot(r, unclipped, lw=1.5, ls=":", label="r*A (kırpılmamış)")
    ax.plot(r, obj, lw=2.5, c="crimson", label="PPO kırpılmış amaç")
    ax.axvspan(1-eps, 1+eps, color="seagreen", alpha=0.12)
    ax.axvline(1.0, c="k", lw=1, ls="--")
    ax.set_xlabel("olasılık oranı r"); ax.set_ylabel("amaç fonksiyonu")
    ax.set_title(name); ax.legend(fontsize=9); ax.grid(alpha=0.3)

# min neden önemli: orana göre gradyan büyüklüğü
grad_pos = np.gradient(np.minimum(r*1.0, np.clip(r, 1-eps, 1+eps)*1.0), r)
grad_neg = np.gradient(np.minimum(r*-1.0, np.clip(r, 1-eps, 1+eps)*-1.0), r)
axes[2].plot(r, grad_pos, lw=2, label="A > 0")
axes[2].plot(r, grad_neg, lw=2, label="A < 0")
axes[2].axvspan(1-eps, 1+eps, color="seagreen", alpha=0.12)
axes[2].set_xlabel("olasılık oranı r"); axes[2].set_ylabel("d amaç / d r")
axes[2].set_title("Gradyan yalnızca 'fazla iyi' tarafta kapatılır")
axes[2].legend(fontsize=9); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print("A>0 ve r>1+eps  -> kırpılır, gradyan yok (zaten yeterince ilerledik).")
print("A<0 ve r>1+eps  -> KIRPILMAZ: politika kötü bir eylemi artırdı, düzeltilmeli.")


## 5. RLHF: Tercihler → Ödül Modeli → KL-Düzenlileştirilmiş Politika

Dil modeli hizalamasında elle yazılmış bir ödül kullanılamaz; bu yüzden bir ödül modeli, insanların
**ikili tercihlerine** Bradley–Terry olabilirliğiyle uydurulur:

$$p(y_w \succ y_l \mid x) = \sigma\big(r_\phi(x, y_w) - r_\phi(x, y_l)\big).$$

Yalnızca ödül *farkları* belirlenebilir — model, $x$'in herhangi bir fonksiyonunun eklenmesine karşı
değişmezdir. Ham ödül değerlerinin istemler arasında karşılaştırılamaz olmasının nedeni budur.

Politika daha sonra referans modele doğru bir KL cezasıyla optimize edilir:

$$\max_\pi \ \mathbb{E}_{y\sim\pi}\big[r(x,y)\big] - \beta\,\text{KL}\big(\pi \,\|\, \pi_{\text{ref}}\big).$$

Bunun **kapalı form çözümü** vardır; fonksiyonel türev sıfıra eşitlenerek elde edilir:

$$\pi^\star(y\mid x) = \frac{1}{Z(x)}\, \pi_{\text{ref}}(y\mid x)\, \exp\!\big(r(x,y)/\beta\big).$$

KL terimi sonradan eklenmiş bir güvenlik önlemi değildir; o olmadan optimum, ödül modelinin en yüksek
puanladığı neyse ona yığılmış bir nokta kütlesidir ve bu neredeyse her zaman ödül modelinin bir
yapaylığıdır.


In [ ]:
# Her şeyin tam hesaplanabilmesi için küçük ayrık bir "yanıt sözlüğü"
K = 12
rng = np.random.default_rng(3)
true_r  = rng.normal(0, 1, K)                     # gözlenmeyen insan kalitesi
pi_ref  = softmax(rng.normal(0, 1, K))            # referans (SFT) politikası

# 1) İkili tercihleri topla ve Bradley-Terry ödül modelini uydur
n_pairs = 4000
i, j = rng.integers(0, K, n_pairs), rng.integers(0, K, n_pairs)
pref = (rng.random(n_pairs) < 1/(1+np.exp(-(true_r[i]-true_r[j])))).astype(float)

r_hat = np.zeros(K)
for _ in range(3000):
    d = r_hat[i] - r_hat[j]
    g = (1/(1+np.exp(-d))) - pref
    grad = np.zeros(K)
    np.add.at(grad, i,  g); np.add.at(grad, j, -g)
    r_hat -= 0.05*grad/n_pairs
r_hat -= r_hat.mean(); true_c = true_r - true_r.mean()      # toplamsal serbestliği sabitle

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].scatter(true_c, r_hat, s=45)
lims = [min(true_c.min(), r_hat.min())-0.3, max(true_c.max(), r_hat.max())+0.3]
axes[0].plot(lims, lims, "k--", lw=1)
axes[0].set_xlabel("gerçek (merkezlenmiş) ödül"); axes[0].set_ylabel("öğrenilen ödül")
axes[0].set_title(f"Bradley-Terry uydurması, korelasyon = {np.corrcoef(true_c, r_hat)[0,1]:.3f}"); axes[0].grid(alpha=0.3)

# 2) Birkaç beta için KL-düzenlileştirilmiş optimum
def optimal_policy(r, beta):
    z = np.log(pi_ref + 1e-12) + r/beta
    return softmax(z)

for beta, c in zip([10.0, 1.0, 0.3, 0.05], ["#c6dbef", "#6baed6", "#2171b5", "#08306b"]):
    axes[1].plot(optimal_policy(r_hat, beta), "o-", lw=2, c=c, label=f"beta={beta}")
axes[1].plot(pi_ref, "k--", lw=2, label="referans politika")
axes[1].set_xlabel("yanıt indeksi"); axes[1].set_ylabel("olasılık")
axes[1].set_title("beta, pi_ref ile argmax r arasında geçiş yapar")
axes[1].legend(fontsize=8); axes[1].grid(alpha=0.3)

betas = np.logspace(-1.5, 1.2, 40)
kls   = [float(np.sum(optimal_policy(r_hat, b)*np.log(optimal_policy(r_hat, b)/pi_ref))) for b in betas]
rews  = [float(optimal_policy(r_hat, b) @ true_c) for b in betas]
axes[2].plot(kls, rews, "o-", lw=2)
axes[2].set_xlabel("KL(pi* || pi_ref)"); axes[2].set_ylabel("gerçek beklenen ödül")
axes[2].set_title("Ödül-KL sınırı"); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print("Ödül modelleri yalnızca farkları belirler: r'ye sabit eklemek pi*'ı değiştirmez.")


## 6. DPO: Ödül Modelini Atlamak

Kapalı form optimumunu tersine çevirin. $\pi^\star \propto \pi_{\text{ref}}\exp(r/\beta)$'dan

$$r(x,y) = \beta \log \frac{\pi^\star(y\mid x)}{\pi_{\text{ref}}(y\mid x)} + \beta \log Z(x)$$

elde edilir. Bunu Bradley–Terry olabilirliğine yerleştirdiğimizde hesaplanamaz $\log Z(x)$ terimi
**sadeleşir**; çünkü aynı isteme verilen iki yanıt arasındaki ödül *farkları* dışında hiçbir şey
görünmez:

$$\mathcal{L}_{\text{DPO}} = -\log \sigma\!\left(\beta \log\frac{\pi_\theta(y_w|x)}{\pi_{\text{ref}}(y_w|x)}
- \beta \log\frac{\pi_\theta(y_l|x)}{\pi_{\text{ref}}(y_l|x)}\right).$$

Politika artık kendi örtük ödül modelidir: tek bir denetimli-benzeri kayıp, örnekleme döngüsü yok,
ayrı bir eleştirmen (critic) yok. Aşağıda DPO aynı tercih verisi üzerinde çalıştırılıp analitik RLHF
optimumuyla karşılaştırılıyor.


In [ ]:
beta = 0.3
sigmoid = lambda z: 1/(1+np.exp(-z))

# DPO: politika logitlerini doğrudan tercih çiftleri üzerinde optimize et
logits = np.log(pi_ref + 1e-12).copy()
log_ref = np.log(pi_ref + 1e-12)
hist = []
for step in range(6000):
    logp = logits - np.logaddexp.reduce(logits)
    yw = np.where(pref > 0.5, i, j)
    yl = np.where(pref > 0.5, j, i)
    margin = beta*((logp[yw]-log_ref[yw]) - (logp[yl]-log_ref[yl]))
    g = -(1 - sigmoid(margin))*beta
    grad = np.zeros(K)
    np.add.at(grad, yw,  g); np.add.at(grad, yl, -g)
    p = np.exp(logp)
    grad = grad/n_pairs
    grad = grad - p*grad.sum()                       # softmax üzerinden izdüşür
    logits -= 0.5*grad
    if step % 50 == 0:
        hist.append(float(np.exp(logp) @ true_c))

pi_dpo  = np.exp(logits - np.logaddexp.reduce(logits))
pi_rlhf = optimal_policy(r_hat, beta)

fig, axes = plt.subplots(1, 3, figsize=(16, 4.2))
axes[0].plot(pi_ref,  "k--", lw=2, label="referans")
axes[0].plot(pi_rlhf, "o-",  lw=2, label="RLHF kapalı form (ödül modeli + KL)")
axes[0].plot(pi_dpo,  "s--", lw=2, label="DPO (ödül modeli yok)")
axes[0].set_xlabel("yanıt indeksi"); axes[0].set_ylabel("olasılık")
axes[0].set_title(f"Aynı optimum, beta={beta}"); axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].plot(np.arange(len(hist))*50, hist, lw=2)
axes[1].axhline(float(pi_rlhf @ true_c), ls="--", c="crimson", label="RLHF optimumu")
axes[1].set_xlabel("DPO adımı"); axes[1].set_ylabel("gerçek beklenen ödül")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3); axes[1].set_title("DPO eğitim eğrisi")

# DPO'nun geri kazandırdığı örtük ödül
r_implicit = beta*(np.log(pi_dpo+1e-12) - log_ref)
r_implicit -= r_implicit.mean()
axes[2].scatter(true_c, r_implicit, s=45, label="DPO örtük ödülü")
axes[2].scatter(true_c, r_hat, s=45, marker="s", alpha=0.7, label="açık ödül modeli")
axes[2].plot(lims, lims, "k--", lw=1)
axes[2].set_xlabel("gerçek ödül"); axes[2].set_ylabel("geri kazanılan ödül")
axes[2].set_title("Politikanın kendisi ödül modelidir"); axes[2].legend(fontsize=8); axes[2].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print(f"toplam değişim |pi_DPO - pi_RLHF| = {0.5*np.abs(pi_dpo-pi_rlhf).sum():.4f}")
print(f"DPO örtük ödülünün gerçekle korelasyonu: {np.corrcoef(true_c, r_implicit)[0,1]:.3f}")


## 7. Ödül Aşırı-Optimizasyonu (Goodhart Yasası)

Ödül modeli bir *vekildir*; sonlu veriden uydurulmuştur. Ona karşı optimize etmek gerçek kaliteyi bir
noktaya kadar artırır, sonra tersine döner: politika, vekilin yüksek olduğu bölgeleri bulur — ama
oralarda yüksek olmasının nedeni çıktının iyi olması değil, vekilin **yanılıyor** olmasıdır.
Deneysel olarak gerçek ödül, $\sqrt{\text{KL}}$'ye göre ters-U çizer ve ödül modeli daha çok veri
gördükçe tepe noktası sağa kayar.

KL bütçesinin birinci sınıf bir hiperparametre olmasının, uygulayıcıların ödül yerine tutulan insan
değerlendirmeleriyle erken durdurma yapmasının ve ödül modeli topluluklarının işe yaramasının nedeni
budur — bağımsız hataların aynı anda sömürülebilir olma olasılığı düşüktür.


In [ ]:
rng = np.random.default_rng(5)
K2 = 400
true_q  = rng.normal(0, 1, K2)
pi_ref2 = softmax(rng.normal(0, 0.5, K2))

def frontier(n_pref_data):
    err = 1.6/np.sqrt(n_pref_data)                       # ödül modeli hatası veriyle küçülür
    proxy = true_q + err*rng.normal(size=K2)
    kls, tru, prx = [], [], []
    for b in np.logspace(1.2, -2.0, 60):
        p = softmax(np.log(pi_ref2+1e-12) + proxy/b)
        kls.append(float(np.sum(p*np.log(p/pi_ref2 + 1e-12))))
        tru.append(float(p @ true_q)); prx.append(float(p @ proxy))
    return np.sqrt(np.array(kls)), np.array(tru), np.array(prx)

fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for n_data, c in zip([25, 100, 1000, 10000], ["#c6dbef", "#6baed6", "#2171b5", "#08306b"]):
    sk, tru, prx = frontier(n_data)
    axes[0].plot(sk, tru, lw=2, c=c, label=f"{n_data} tercih")
    axes[0].scatter([sk[int(np.argmax(tru))]], [tru.max()], c=c, s=45, zorder=4)
axes[0].set_xlabel("sqrt KL(pi || pi_ref)"); axes[0].set_ylabel("GERÇEK beklenen ödül")
axes[0].set_title("Aşırı-optimizasyon: fazla KL sonunda zarar verir")
axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

sk, tru, prx = frontier(100)
axes[1].plot(sk, prx, lw=2, label="vekil ödül (optimize ettiğiniz)")
axes[1].plot(sk, tru, lw=2, label="gerçek ödül (istediğiniz)")
axes[1].axvline(sk[int(np.argmax(tru))], ls="--", c="crimson", label="optimal durma noktası")
axes[1].set_xlabel("sqrt KL"); axes[1].set_title("Gerçek düşüşe geçtikten sonra vekil yükselmeye devam ediyor")
axes[1].legend(fontsize=9); axes[1].grid(alpha=0.3)

plt.tight_layout(); plt.show()
print("Vekil ödül, kuruluşu gereği KL'de monotondur -- ne zaman duracağınızı asla söyleyemez.")


## 8. Özet

| Kavram | Açıklama |
|---|---|
| **Ölümcül üçlü** | Fonksiyon yaklaşımı + önyükleme + politika-dışı ⇒ ıraksama mümkün |
| **Yarı-gradyan TD** | Hiçbir amaç fonksiyonunun gradyanı değildir; hedef ağlar ve tekrar belleği hafifletir |
| **Aşırı tahmin yanlılığı** | $\mathbb{E}[\max \hat Q] \ge \max \mathbb{E}[\hat Q]$; eylem ve gürültüyle büyür |
| **Çift Q-öğrenme** | *Seçim* ve *değerlendirme* kestiricilerini ayırır |
| **Temel değerler** | Herhangi bir $b(s)$ gradyanı yansız bırakır; $b \approx V(s)$ varyansı minimize eder |
| **GAE($\lambda$)** | Üstel ağırlıklı avantajlar; $\lambda$ yanlılığı varyansla takas eder |
| **Güven bölgesi** | Vekil amaç yalnızca $\pi_{\text{eski}}$ yakınında geçerlidir |
| **PPO kırpma** | $[1-\epsilon, 1+\epsilon]$ dışında düz amaç; $\min$ onu kötümser tutar |
| **Bradley–Terry** | $\sigma(r_w - r_l)$; yalnızca ödül farkları belirlenebilir |
| **RLHF optimumu** | $\pi^\star \propto \pi_{\text{ref}}\exp(r/\beta)$ |
| **DPO** | Bu optimumu BT'ye yerleştirir; $\log Z(x)$ sadeleşir; ödül modeli gerekmez |
| **Aşırı-optimizasyon** | Gerçek ödül $\sqrt{\text{KL}}$'de ters-U çizer; vekil sizi asla uyarmaz |

**Sonraki Defter →** Geometrik Derin Öğrenme ve İfade Gücü
